## Data Analysis Using SQL and Pandas on **Chinkook Database**

### Importing Libraries

In [199]:
import pandas as pd
import sqlite3

### Connecting To my Sqlite file


In [200]:
conn = sqlite3.connect('Chinook_Sqlite.sqlite')
cursor = conn.cursor()

### Sanity Checking That All Tables are Loaded

In [201]:
pd.read_sql_query("SELECT name FROM sqlite_master WHERE type = 'table';",conn)

,name
0,Album
1,Artist
2,Customer
3,Employee
4,Genre
5,Invoice
6,InvoiceLine
7,MediaType
8,Playlist
9,PlaylistTrack


### Top **5** Genres With the Highest Revenue

In [202]:
pd.read_sql_query("SELECT g.GenreId,g.Name,SUM(il.UnitPrice*il.Quantity) AS total_revenue FROM Genre g JOIN Track t ON g.GenreId = t.GenreId JOIN InvoiceLine il ON il.TrackId = t.TrackId GROUP BY g.GenreID,g.Name ORDER BY total_revenue DESC LIMIT 5",conn)

,GenreId,Name,total_revenue
0,1,Rock,826.65
1,7,Latin,382.14
2,3,Metal,261.36
3,4,Alternative & Punk,241.56
4,19,TV Shows,93.53


##### This Table shows us that **Rock** has the highest revenue of *826.65 dollars*.

### Top **5** customers by Total Amount Spent

In [203]:
pd.read_sql_query("SELECT c.CustomerId,c.FirstName,c.LastName,SUM(i.Total) AS Total_Amount_Spent FROM Customer c JOIN Invoice i ON c.CustomerId = i.CustomerId GROUP BY c.CustomerId,c.FirstName,c.LastName ORDER BY Total_Amount_Spent DESC LIMIT 5",conn)

,CustomerId,FirstName,LastName,Total_Amount_Spent
0,6,Helena,Holý,49.62
1,26,Richard,Cunningham,47.62
2,57,Luis,Rojas,46.62
3,45,Ladislav,Kovács,45.62
4,46,Hugh,O'Reilly,45.62


##### CustomerId's **6, 26, 57, 45, 46** spent the most Total Amount with *Helena Holý* spent the most amount of **49.62**

### Artist with the most tracks

In [204]:
pd.read_sql_query("SELECT Artist.ArtistId,Artist.Name,COUNT(Track.TrackId) AS Total_Tracks FROM Artist JOIN Album ON Artist.ArtistId = Album.ArtistId JOIN Track ON Track.AlbumId = Album.AlbumId GROUP BY Artist.ArtistId,Artist.Name ORDER BY Total_Tracks DESC LIMIT 1",conn)

,ArtistId,Name,Total_Tracks
0,90,Iron Maiden,213


#### **Iron Maiden** is the artist with the most number of tracks of *213*

### Sales per Month

In [205]:
pd.read_sql_query("SELECT strftime('%Y-%m', i.InvoiceDate) AS Year_Months,SUM(il.Quantity*il.UnitPrice) AS sales_per_month FROM Invoice i JOIN InvoiceLine il ON i.InvoiceId = il.InvoiceId GROUP BY Year_Months ORDER BY Year_Months ",conn)

,Year_Months,sales_per_month
0,2009-01,35.64
1,2009-02,37.62
2,2009-03,37.62
3,2009-04,37.62
4,2009-05,37.62
5,2009-06,37.62
6,2009-07,37.62
7,2009-08,37.62
8,2009-09,37.62
9,2009-10,37.62


### Country with the Highest Revenue

In [206]:
pd.read_sql_query("SELECT BillingCountry AS Country,SUM(Total) AS Revenue FROM Invoice GROUP BY BillingCountry ORDER BY Revenue DESC",conn)


,Country,Revenue
0,USA,523.06
1,Canada,303.96
2,France,195.10
3,Brazil,190.10
4,Germany,156.48
5,United Kingdom,112.86
6,Czech Republic,90.24
7,Portugal,77.24
8,India,75.26
9,Chile,46.62


##### **USA** has the highest revenue of *523.06 dollars*

### Above Average Invoice and Average Invoice

In [207]:
pd.read_sql("SELECT InvoiceId,CustomerId,strftime('%Y-%m-%d',InvoiceDate) AS Dates,(SELECT AVG(Total) FROM Invoice) AS Overall_Average,Total FROM Invoice WHERE Total >(SELECT AVG(Total) AS Average_Invoice_Value FROM Invoice) ORDER BY Total Desc;",conn)



,InvoiceId,CustomerId,Dates,Overall_Average,Total
0,404,6,2013-11-13,5.651942,25.86
1,299,26,2012-08-05,5.651942,23.86
2,96,45,2010-02-18,5.651942,21.86
3,194,46,2011-04-28,5.651942,21.86
4,89,7,2010-01-18,5.651942,18.86
...,...,...,...,...,...
174,381,54,2013-08-04,5.651942,5.94
175,388,33,2013-09-04,5.651942,5.94
176,395,12,2013-10-05,5.651942,5.94
177,402,50,2013-11-05,5.651942,5.94


##### **Invoice 404** has the highest total average of *25.89 dollars*. And an **Overall Average** of *5.65 dollars*

### Employees with the highest total sales through their supported customers

In [208]:
pd.read_sql_query("SELECT e.EmployeeId AS EmployeeID,e.FirstName AS Employee_First_Name,e.LastName AS Employee_Last_Name,SUM(i.Total) AS Total_Sales FROM Employee e JOIN Customer c ON e.EmployeeId = c.SupportRepId JOIN Invoice i ON i.CustomerId = c.CustomerId GROUP BY e.EmployeeId,e.FirstName,e.LastName ORDER BY total_sales DESC",conn)

,EmployeeID,Employee_First_Name,Employee_Last_Name,Total_Sales
0,3,Jane,Peacock,833.04
1,4,Margaret,Park,775.40
2,5,Steve,Johnson,720.16


##### **Employee 3 Jane Peacock** has the Highest Total Sales of *833.04 dollars*

### Never Purchased Tracks

In [209]:
pd.read_sql_query("SELECT t.TrackId,t.Name FROM Track t LEFT JOIN InvoiceLine il ON t.TrackId = il.TrackId WHERE il.TrackId IS NULL",conn)

,TrackId,Name
0,7,Let's Get It Up
1,11,C.O.D.
2,17,Let There Be Rock
3,18,Bad Boy Boogie
4,22,Whole Lotta Rosie
...,...,...
1514,3497,"Erlkonig, D.328"
1515,3498,"Concerto for Violin, Strings and Continuo in G..."
1516,3501,"L'orfeo, Act 3, Sinfonia (Orchestra)"
1517,3502,"Quintet for Horn, Violin, 2 Violas, and Cello ..."


##### **1519** Tracks were never purchased even once

### Customers with only a Single Purchase

In [210]:
pd.read_sql_query("SELECT c.CustomerId,c.FirstName AS Customer_First_Name,c.LastName AS Customer_Last_Name,COUNT(i.InvoiceId) AS Total_Counts FROM Customer c JOIN Invoice i ON c.CustomerId = i.CustomerId GROUP BY c.CustomerId,c.FirstName,c.LastName HAVING Total_Counts = 1",conn)


,CustomerId,Customer_First_Name,Customer_Last_Name,Total_Counts


##### No customers with only a Single Purchase. Chinook only has 59 customers and 412 invoices, so most customers naturally have several orders

### Most popular media type (MP3, AAC, etc.) by Number of Tracks sold

In [211]:
pd.read_sql_query("SELECT mt.Name AS MediaTypeName, COUNT(il.InvoiceLineId) AS TracksSold FROM MediaType mt JOIN Track t ON mt.MediaTypeId = t.MediaTypeId JOIN InvoiceLine il ON t.TrackId = il.TrackId GROUP BY mt.MediaTypeId, mt.Name ORDER BY TracksSold DESC;",conn)

,MediaTypeName,TracksSold
0,MPEG audio file,1976
1,Protected AAC audio file,146
2,Protected MPEG-4 video file,111
3,Purchased AAC audio file,4
4,AAC audio file,3


**MPEG audio file** is the most popular Media Type with *1976 Track solds*